<a href="https://colab.research.google.com/github/sulucay01/DI725-assignment1/blob/dev/notebooks/04_modeling_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04_modeling_fusion.ipynb

## Part 1: Categorical Feature Fusion

#### Imports and config

In [1]:
import os
import re
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

from torch.utils.data import Dataset

#### Reset Seed

In [2]:
def reset_all_seeds(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

#### Config

In [3]:
# =========================
# CONFIG
# =========================
SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
OVERSAMPLE_FACTOR = 3

WANDB_PROJECT = "di725-sentiment"

SPECIAL_TOKENS = ["[CUSTOMER]", "[AGENT]", "[EMAIL]", "[PHONE]", "[ID]"]

CATEGORICAL_COLS = ["issue_area", "issue_category"]

# You can change these later if you want
CAT_EMB_DIM = 32
NUM_HIDDEN_DIM = 64
FUSION_HIDDEN_DIM = 256
DROPOUT = 0.2

LR = 2e-5
EPOCHS = 3
TRAIN_BS = 8
EVAL_BS = 16

reset_all_seeds(SEED)

#### Load data

In [4]:
train_df = pd.read_csv("https://raw.githubusercontent.com/sulucay01/DI725-assignment1/dev/data/processed/train_processed.csv")
val_df   = pd.read_csv("https://raw.githubusercontent.com/sulucay01/DI725-assignment1/dev/data/processed/val_processed.csv")
test_df  = pd.read_csv("https://raw.githubusercontent.com/sulucay01/DI725-assignment1/dev/data/processed/test_processed.csv")

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (773, 16)
val:   (194, 16)
test:  (30, 16)


In [5]:
for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["clean_text"].fillna("").astype(str)
    df["issue_area"] = df["issue_area"].fillna("UNKNOWN").astype(str)
    df["issue_category"] = df["issue_category"].fillna("UNKNOWN").astype(str)

#### Encode labels

In [6]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["customer_sentiment"])
val_df["label"]   = label_encoder.transform(val_df["customer_sentiment"])
test_df["label"]  = label_encoder.transform(test_df["customer_sentiment"])

label_mapping = {cls: int(i) for i, cls in enumerate(label_encoder.classes_)}
print("Label mapping:", label_mapping)
print(train_df["customer_sentiment"].value_counts())

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
customer_sentiment
neutral     433
negative    326
positive     14
Name: count, dtype: int64


#### Encode categorical features

In [7]:
cat_encoders = {}

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(train_df[col])

    known_classes = set(le.classes_)

    if "UNKNOWN" not in known_classes:
        le.classes_ = np.append(le.classes_, "UNKNOWN")
        known_classes = set(le.classes_)

    train_df[col] = train_df[col].apply(lambda x: x if x in known_classes else "UNKNOWN")
    val_df[col]   = val_df[col].apply(lambda x: x if x in known_classes else "UNKNOWN")
    test_df[col]  = test_df[col].apply(lambda x: x if x in known_classes else "UNKNOWN")

    train_df[f"{col}_id"] = le.transform(train_df[col])
    val_df[f"{col}_id"]   = le.transform(val_df[col])
    test_df[f"{col}_id"]  = le.transform(test_df[col])

    cat_encoders[col] = le

print("issue_area classes:", len(cat_encoders["issue_area"].classes_))
print("issue_category classes:", len(cat_encoders["issue_category"].classes_))

issue_area classes: 7
issue_category classes: 41


#### Tokenizer and tail truncation

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

def truncate_text_tail(text, tokenizer, max_length=512):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    budget = max_length - 2

    if len(token_ids) <= budget:
        return text

    truncated_ids = token_ids[-budget:]
    return tokenizer.decode(
        truncated_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=True
    )

for df in [train_df, val_df, test_df]:
    df["model_text"] = df["clean_text"].apply(
        lambda x: truncate_text_tail(x, tokenizer, max_length=MAX_LENGTH)
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (514 > 512). Running this sequence through the model will result in indexing errors


#### Create numerical features

In [9]:
def safe_div(a, b):
    return a / b if b != 0 else 0.0

def split_turns(text):
    """
    Splits text into speaker turns using [CUSTOMER] and [AGENT] markers.

    Returns:
        List of tuples: [(speaker_token, utterance), ...]
    """
    if not isinstance(text, str):
        return []

    pattern = r"(\[CUSTOMER\]|\[AGENT\])"
    parts = re.split(pattern, text)

    turns = []
    current_speaker = None

    for part in parts:
        if part in ["[CUSTOMER]", "[AGENT]"]:
            current_speaker = part
        else:
            utterance = part.strip()
            if current_speaker is not None and utterance != "":
                turns.append((current_speaker, utterance))

    return turns


def extract_numeric_features(df, text_col="model_text"):
    out = df.copy()
    text_series = out[text_col].fillna("").astype(str)

    # basic text size
    out["num_lines"] = text_series.apply(
        lambda x: len([line for line in x.split("\n") if line.strip() != ""])
    )
    out["num_words"] = text_series.apply(lambda x: len(x.split()))
    out["num_chars"] = text_series.str.len()
    out["avg_word_len"] = text_series.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0.0
    )

    # punctuation / style
    out["question_mark_count"] = text_series.str.count(r"\?")
    out["exclamation_count"] = text_series.str.count(r"!")
    out["digit_count"] = text_series.str.count(r"\d")

    # special token counts
    out["email_token_count"] = text_series.apply(lambda x: x.count("[EMAIL]"))
    out["phone_token_count"] = text_series.apply(lambda x: x.count("[PHONE]"))
    out["id_token_count"] = text_series.apply(lambda x: x.count("[ID]"))
    out["customer_token_count"] = text_series.apply(lambda x: x.count("[CUSTOMER]"))
    out["agent_token_count"] = text_series.apply(lambda x: x.count("[AGENT]"))

    out["sensitive_token_total"] = (
        out["email_token_count"] +
        out["phone_token_count"] +
        out["id_token_count"]
    )
    out["sensitive_token_ratio"] = out.apply(
        lambda row: safe_div(row["sensitive_token_total"], row["num_words"]),
        axis=1
    )

    # turn-based features
    turns_all = text_series.apply(split_turns)

    out["total_turn_count"] = turns_all.apply(len)
    out["customer_turn_count"] = turns_all.apply(
        lambda turns: sum(1 for speaker, _ in turns if speaker == "[CUSTOMER]")
    )
    out["agent_turn_count"] = turns_all.apply(
        lambda turns: sum(1 for speaker, _ in turns if speaker == "[AGENT]")
    )

    out["customer_turn_ratio"] = out.apply(
        lambda row: safe_div(row["customer_turn_count"], row["total_turn_count"]),
        axis=1
    )
    out["agent_turn_ratio"] = out.apply(
        lambda row: safe_div(row["agent_turn_count"], row["total_turn_count"]),
        axis=1
    )
    out["turn_count_diff"] = out["customer_turn_count"] - out["agent_turn_count"]

    # per-turn word stats
    def turn_word_lengths(turns):
        return [len(utt.split()) for _, utt in turns if utt.strip() != ""]

    turn_lengths = turns_all.apply(turn_word_lengths)

    out["avg_words_per_turn"] = turn_lengths.apply(
        lambda arr: float(np.mean(arr)) if len(arr) > 0 else 0.0
    )
    out["std_turn_len_words"] = turn_lengths.apply(
        lambda arr: float(np.std(arr)) if len(arr) > 0 else 0.0
    )
    out["max_words_in_turn"] = turn_lengths.apply(
        lambda arr: max(arr) if len(arr) > 0 else 0
    )
    out["min_words_in_turn"] = turn_lengths.apply(
        lambda arr: min(arr) if len(arr) > 0 else 0
    )

    # customer / agent text volume
    def speaker_word_count(turns, target_speaker):
        return sum(len(utt.split()) for speaker, utt in turns if speaker == target_speaker)

    def speaker_char_count(turns, target_speaker):
        return sum(len(utt) for speaker, utt in turns if speaker == target_speaker)

    out["customer_word_count"] = turns_all.apply(
        lambda turns: speaker_word_count(turns, "[CUSTOMER]")
    )
    out["agent_word_count"] = turns_all.apply(
        lambda turns: speaker_word_count(turns, "[AGENT]")
    )
    out["customer_char_count"] = turns_all.apply(
        lambda turns: speaker_char_count(turns, "[CUSTOMER]")
    )
    out["agent_char_count"] = turns_all.apply(
        lambda turns: speaker_char_count(turns, "[AGENT]")
    )

    out["customer_to_agent_word_ratio"] = out.apply(
        lambda row: safe_div(row["customer_word_count"], row["agent_word_count"]),
        axis=1
    )
    out["customer_to_agent_char_ratio"] = out.apply(
        lambda row: safe_div(row["customer_char_count"], row["agent_char_count"]),
        axis=1
    )

    out["avg_words_per_customer_turn"] = out.apply(
        lambda row: safe_div(row["customer_word_count"], row["customer_turn_count"]),
        axis=1
    )
    out["avg_words_per_agent_turn"] = out.apply(
        lambda row: safe_div(row["agent_word_count"], row["agent_turn_count"]),
        axis=1
    )

    # ending features
    out["ends_with_customer"] = turns_all.apply(
        lambda turns: 1 if len(turns) > 0 and turns[-1][0] == "[CUSTOMER]" else 0
    )
    out["ends_with_agent"] = turns_all.apply(
        lambda turns: 1 if len(turns) > 0 and turns[-1][0] == "[AGENT]" else 0
    )
    out["last_turn_word_count"] = turns_all.apply(
        lambda turns: len(turns[-1][1].split()) if len(turns) > 0 else 0
    )

    # lexical diversity
    out["unique_word_count"] = text_series.apply(
        lambda x: len(set(x.split()))
    )
    out["lexical_diversity"] = out.apply(
        lambda row: safe_div(row["unique_word_count"], row["num_words"]),
        axis=1
    )

    return out

In [10]:
train_df = extract_numeric_features(train_df, text_col="model_text")
val_df   = extract_numeric_features(val_df, text_col="model_text")
test_df  = extract_numeric_features(test_df, text_col="model_text")

In [11]:
NUMERIC_FEATURE_GROUPS = {
    "basic": [
        "num_lines",
        "num_words",
        "num_chars",
        "avg_word_len"
    ],

    "punctuation_style": [
        "question_mark_count",
        "exclamation_count",
        "digit_count"
    ],

    "special_tokens": [
        "email_token_count",
        "phone_token_count",
        "id_token_count",
        "customer_token_count",
        "agent_token_count",
        "sensitive_token_total",
        "sensitive_token_ratio"
    ],

    "turn_counts": [
        "total_turn_count",
        "customer_turn_count",
        "agent_turn_count",
        "customer_turn_ratio",
        "agent_turn_ratio",
        "turn_count_diff"
    ],

    "turn_length_stats": [
        "avg_words_per_turn",
        "std_turn_len_words",
        "max_words_in_turn",
        "min_words_in_turn"
    ],

    "speaker_volume": [
        "customer_word_count",
        "agent_word_count",
        "customer_char_count",
        "agent_char_count",
        "customer_to_agent_word_ratio",
        "customer_to_agent_char_ratio",
        "avg_words_per_customer_turn",
        "avg_words_per_agent_turn"
    ],

    "ending_features": [
        "ends_with_customer",
        "ends_with_agent",
        "last_turn_word_count"
    ],

    "lexical": [
        "unique_word_count",
        "lexical_diversity"
    ]
}

In [12]:
SELECTED_NUMERIC_COLS = [
    "num_words", "lexical_diversity", "agent_word_count",
    "customer_word_count", "avg_words_per_turn", "customer_turn_count",
    "std_turn_len_words", "exclamation_count"
]

#### Standardize numerical features using train only

In [14]:
scaler = StandardScaler()

train_df[SELECTED_NUMERIC_COLS] = scaler.fit_transform(train_df[SELECTED_NUMERIC_COLS])
val_df[SELECTED_NUMERIC_COLS]   = scaler.transform(val_df[SELECTED_NUMERIC_COLS])
test_df[SELECTED_NUMERIC_COLS]  = scaler.transform(test_df[SELECTED_NUMERIC_COLS])

#### Oversample positive class x3

In [15]:
def oversample_positive_class(train_data, factor=3, seed=42):
    positive_label_id = train_data.loc[
        train_data["customer_sentiment"].str.lower() == "positive", "label"
    ].iloc[0]

    train_non_positive = train_data[train_data["label"] != positive_label_id].copy()
    train_positive = train_data[train_data["label"] == positive_label_id].copy()

    train_positive_os = resample(
        train_positive,
        replace=True,
        n_samples=len(train_positive) * factor,
        random_state=seed
    )

    train_os = pd.concat([train_non_positive, train_positive_os], axis=0)
    train_os = train_os.sample(frac=1, random_state=seed).reset_index(drop=True)

    return train_os

In [16]:
train_os_df = oversample_positive_class(train_df.copy(), factor=OVERSAMPLE_FACTOR, seed=SEED)

print("Original train label distribution:")
print(train_df["customer_sentiment"].value_counts())

print("\nOversampled train label distribution:")
print(train_os_df["customer_sentiment"].value_counts())

Original train label distribution:
customer_sentiment
neutral     433
negative    326
positive     14
Name: count, dtype: int64

Oversampled train label distribution:
customer_sentiment
neutral     433
negative    326
positive     42
Name: count, dtype: int64


#### Tokenize text

In [17]:
def tokenize_dataframe_text(df, tokenizer, max_length=512):
    return tokenizer(
        df["model_text"].tolist(),
        truncation=True,
        max_length=max_length,
        padding=False
    )

train_encodings = tokenize_dataframe_text(train_os_df, tokenizer, MAX_LENGTH)
val_encodings   = tokenize_dataframe_text(val_df, tokenizer, MAX_LENGTH)
test_encodings  = tokenize_dataframe_text(test_df, tokenizer, MAX_LENGTH)

#### Dataset class for flexible fusion

In [18]:
class FlexibleFusionDataset(Dataset):
    def __init__(self, encodings, labels, cat_features=None, num_features=None):
        self.encodings = encodings
        self.labels = labels.astype(np.int64)
        self.cat_features = cat_features.astype(np.int64) if cat_features is not None else None
        self.num_features = num_features.astype(np.float32) if num_features is not None else None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

        if self.cat_features is not None:
            item["cat_features"] = self.cat_features[idx]

        if self.num_features is not None:
            item["num_features"] = self.num_features[idx]

        return item

#### Data collator

In [19]:
class FlexibleFusionCollator:
    def __init__(self, tokenizer):
        self.base_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def __call__(self, features):
        text_features = [
            {
                "input_ids": f["input_ids"],
                "attention_mask": f["attention_mask"]
            }
            for f in features
        ]

        batch = self.base_collator(text_features)

        batch["labels"] = torch.tensor(
            [f["labels"] for f in features],
            dtype=torch.long
        )

        if "cat_features" in features[0]:
            batch["cat_features"] = torch.tensor(
                [f["cat_features"] for f in features],
                dtype=torch.long
            )

        if "num_features" in features[0]:
            batch["num_features"] = torch.tensor(
                [f["num_features"] for f in features],
                dtype=torch.float
            )

        return batch

#### Class weights

In [20]:
class_labels = np.sort(train_df["label"].unique())

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=train_df["label"].values
)

class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float)
print("Class weights:", class_weights_tensor)

Class weights: tensor([ 0.7904,  0.5951, 18.4048])


#### Flexible model

In [21]:
class RobertaFlexibleFusionClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels,
        class_weights,
        tokenizer_len,
        use_categorical=False,
        use_numerical=False,
        issue_area_cardinality=None,
        issue_category_cardinality=None,
        num_numeric_features=None,
        cat_emb_dim=32,
        num_hidden_dim=64,
        fusion_hidden_dim=256,
        dropout=0.2
    ):
        super().__init__()

        self.use_categorical = use_categorical
        self.use_numerical = use_numerical

        self.text_encoder = AutoModel.from_pretrained(model_name)
        self.text_encoder.resize_token_embeddings(tokenizer_len)

        text_hidden_size = self.text_encoder.config.hidden_size
        extra_dim = 0

        if self.use_categorical:
            self.issue_area_embedding = nn.Embedding(issue_area_cardinality, cat_emb_dim)
            self.issue_category_embedding = nn.Embedding(issue_category_cardinality, cat_emb_dim)
            extra_dim += 2 * cat_emb_dim

        if self.use_numerical:
            self.num_projection = nn.Sequential(
                nn.Linear(num_numeric_features, num_hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            )
            extra_dim += num_hidden_dim

        fusion_input_dim = text_hidden_size + extra_dim

        self.fusion_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_input_dim, fusion_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, num_labels)
        )

        self.class_weights = class_weights

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        cat_features=None,
        num_features=None,
        labels=None
    ):
        outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_repr = outputs.last_hidden_state[:, 0, :]

        fusion_parts = [text_repr]

        if self.use_categorical:
            issue_area_ids = cat_features[:, 0]
            issue_category_ids = cat_features[:, 1]

            issue_area_repr = self.issue_area_embedding(issue_area_ids)
            issue_category_repr = self.issue_category_embedding(issue_category_ids)

            fusion_parts.extend([issue_area_repr, issue_category_repr])

        if self.use_numerical:
            num_repr = self.num_projection(num_features)
            fusion_parts.append(num_repr)

        fused_repr = torch.cat(fusion_parts, dim=1)
        logits = self.fusion_head(fused_repr)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

#### Metrics

In [22]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

#### Evaluation helpers

In [23]:
def evaluate_split(trainer, dataset, split_name, label_encoder):
    preds_output = trainer.predict(dataset)
    logits = preds_output.predictions
    y_true = preds_output.label_ids
    y_pred = np.argmax(logits, axis=1)

    label_names = list(label_encoder.classes_)

    print(f"\n{'='*60}")
    print(f"{split_name.upper()} CLASSIFICATION REPORT")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{split_name.upper()} CONFUSION MATRIX")
    print(cm)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": cm
    }

In [24]:
def get_positive_class_metrics(y_true, y_pred, label_encoder, positive_label="positive"):
    pos_idx = list(label_encoder.classes_).index(positive_label)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[pos_idx],
        average=None,
        zero_division=0
    )

    return {
        "precision": precision[0],
        "recall": recall[0],
        "f1": f1[0]
    }

#### Main experiment runner

In [25]:
def run_experiment(
    experiment_name,
    use_categorical=False,
    use_numerical=False
):
    reset_all_seeds(SEED)

    wandb.init(
        project=WANDB_PROJECT,
        name=experiment_name,
        reinit=True,
        config={
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "oversample_factor": OVERSAMPLE_FACTOR,
            "use_categorical": use_categorical,
            "use_numerical": use_numerical,
            "categorical_features": CATEGORICAL_COLS if use_categorical else [],
            "numerical_features": SELECTED_NUMERIC_COLS if use_numerical else [],
            "cat_emb_dim": CAT_EMB_DIM,
            "num_hidden_dim": NUM_HIDDEN_DIM,
            "fusion_hidden_dim": FUSION_HIDDEN_DIM,
            "dropout": DROPOUT,
            "learning_rate": LR,
            "epochs": EPOCHS,
            "train_batch_size": TRAIN_BS,
            "eval_batch_size": EVAL_BS,
            "truncation": "tail",
            "special_tokens": SPECIAL_TOKENS
        }
    )

    # Build arrays
    train_cat = train_os_df[["issue_area_id", "issue_category_id"]].values if use_categorical else None
    val_cat   = val_df[["issue_area_id", "issue_category_id"]].values if use_categorical else None
    test_cat  = test_df[["issue_area_id", "issue_category_id"]].values if use_categorical else None

    train_num = train_os_df[SELECTED_NUMERIC_COLS].values if use_numerical else None
    val_num   = val_df[SELECTED_NUMERIC_COLS].values if use_numerical else None
    test_num  = test_df[SELECTED_NUMERIC_COLS].values if use_numerical else None

    train_dataset = FlexibleFusionDataset(
        encodings=train_encodings,
        labels=train_os_df["label"].values,
        cat_features=train_cat,
        num_features=train_num
    )

    val_dataset = FlexibleFusionDataset(
        encodings=val_encodings,
        labels=val_df["label"].values,
        cat_features=val_cat,
        num_features=val_num
    )

    test_dataset = FlexibleFusionDataset(
        encodings=test_encodings,
        labels=test_df["label"].values,
        cat_features=test_cat,
        num_features=test_num
    )

    model = RobertaFlexibleFusionClassifier(
        model_name=MODEL_NAME,
        num_labels=len(class_labels),
        class_weights=class_weights_tensor,
        tokenizer_len=len(tokenizer),
        use_categorical=use_categorical,
        use_numerical=use_numerical,
        issue_area_cardinality=len(cat_encoders["issue_area"].classes_) if use_categorical else None,
        issue_category_cardinality=len(cat_encoders["issue_category"].classes_) if use_categorical else None,
        num_numeric_features=len(SELECTED_NUMERIC_COLS) if use_numerical else None,
        cat_emb_dim=CAT_EMB_DIM,
        num_hidden_dim=NUM_HIDDEN_DIM,
        fusion_hidden_dim=FUSION_HIDDEN_DIM,
        dropout=DROPOUT
    )

    training_args = TrainingArguments(
        output_dir=f"./outputs/{experiment_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=20,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="wandb",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=FlexibleFusionCollator(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    val_metrics = trainer.evaluate()
    print("\nValidation metrics:", val_metrics)

    train_results = evaluate_split(trainer, train_dataset, "train", label_encoder)
    val_results   = evaluate_split(trainer, val_dataset, "val", label_encoder)
    test_results  = evaluate_split(trainer, test_dataset, "test", label_encoder)

    train_pos = get_positive_class_metrics(train_results["y_true"], train_results["y_pred"], label_encoder)
    val_pos   = get_positive_class_metrics(val_results["y_true"], val_results["y_pred"], label_encoder)
    test_pos  = get_positive_class_metrics(test_results["y_true"], test_results["y_pred"], label_encoder)

    wandb.log({
        "final/train_positive_precision": train_pos["precision"],
        "final/train_positive_recall": train_pos["recall"],
        "final/train_positive_f1": train_pos["f1"],
        "final/val_positive_precision": val_pos["precision"],
        "final/val_positive_recall": val_pos["recall"],
        "final/val_positive_f1": val_pos["f1"],
        "final/test_positive_precision": test_pos["precision"],
        "final/test_positive_recall": test_pos["recall"],
        "final/test_positive_f1": test_pos["f1"],
    })

    wandb.finish()

    return {
        "experiment_name": experiment_name,
        "val_metrics": val_metrics,
        "train_results": train_results,
        "val_results": val_results,
        "test_results": test_results,
        "train_positive": train_pos,
        "val_positive": val_pos,
        "test_positive": test_pos
    }

#### Run experiment 1: text + categorical

In [26]:
result_text_cat = run_experiment(
    experiment_name="roberta_text_categorical_os3",
    use_categorical=True,
    use_numerical=False
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: selingoktas98 (selingoktas98-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, 

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.927241,0.902307,0.670103,0.444389,0.663266
2,0.564390,0.567316,0.819588,0.661093,0.829085
3,0.526028,0.479549,0.835052,0.568354,0.839443



Validation metrics: {'eval_loss': 0.5673155784606934, 'eval_accuracy': 0.8195876288659794, 'eval_macro_f1': 0.6610929816917565, 'eval_weighted_f1': 0.8290850938283391, 'eval_runtime': 0.3141, 'eval_samples_per_second': 617.568, 'eval_steps_per_second': 41.383, 'epoch': 3.0}

TRAIN CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.86      0.83      0.85       326
     neutral       0.85      0.82      0.84       433
    positive       0.50      0.79      0.61        42

    accuracy                           0.83       801
   macro avg       0.74      0.81      0.76       801
weighted avg       0.84      0.83      0.83       801


TRAIN CONFUSION MATRIX
[[272  54   0]
 [ 44 356  33]
 [  0   9  33]]



VAL CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.84      0.84      0.84        82
     neutral       0.86      0.81      0.83       109
    positive       0.20      0.67      0.31         3

    accuracy                           0.82       194
   macro avg       0.63      0.77      0.66       194
weighted avg       0.84      0.82      0.83       194


VAL CONFUSION MATRIX
[[69 13  0]
 [13 88  8]
 [ 0  1  2]]



TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.90      0.90      0.90        10
     neutral       0.60      0.90      0.72        10
    positive       1.00      0.50      0.67        10

    accuracy                           0.77        30
   macro avg       0.83      0.77      0.76        30
weighted avg       0.83      0.77      0.76        30


TEST CONFUSION MATRIX
[[9 1 0]
 [1 9 0]
 [0 5 5]]


eval/accuracy,▁▇█▇
eval/loss,█▂▁▂
eval/macro_f1,▁█▅█
eval/runtime,▂▁█▅
eval/samples_per_second,▆█▁▃
eval/steps_per_second,▆█▁▃
eval/weighted_f1,▁███
final/test_positive_f1,▁
final/test_positive_precision,▁
final/test_positive_recall,▁
+18,...


#### Run experiment 1: text + numerical

In [27]:
result_text_num = run_experiment(
    experiment_name="roberta_text_numerical_os3",
    use_categorical=False,
    use_numerical=True
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.019341,0.926812,0.623711,0.339413,0.534410
2,0.595991,0.525452,0.737113,0.571347,0.769542
3,0.544160,0.452194,0.773196,0.599640,0.806876



Validation metrics: {'eval_loss': 0.45219382643699646, 'eval_accuracy': 0.7731958762886598, 'eval_macro_f1': 0.5996397910024119, 'eval_weighted_f1': 0.8068763557782602, 'eval_runtime': 0.3164, 'eval_samples_per_second': 613.144, 'eval_steps_per_second': 41.087, 'epoch': 3.0}

TRAIN CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.87      0.89      0.88       326
     neutral       0.89      0.76      0.82       433
    positive       0.38      0.88      0.53        42

    accuracy                           0.82       801
   macro avg       0.71      0.84      0.74       801
weighted avg       0.85      0.82      0.83       801


TRAIN CONFUSION MATRIX
[[291  35   0]
 [ 45 328  60]
 [  0   5  37]]



VAL CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.89      0.83      0.86        82
     neutral       0.84      0.73      0.78       109
    positive       0.09      0.67      0.15         3

    accuracy                           0.77       194
   macro avg       0.61      0.74      0.60       194
weighted avg       0.85      0.77      0.81       194


VAL CONFUSION MATRIX
[[68 14  0]
 [ 8 80 21]
 [ 0  1  2]]



TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.80      0.80      0.80        10
     neutral       0.64      0.70      0.67        10
    positive       0.89      0.80      0.84        10

    accuracy                           0.77        30
   macro avg       0.78      0.77      0.77        30
weighted avg       0.78      0.77      0.77        30


TEST CONFUSION MATRIX
[[8 2 0]
 [2 7 1]
 [0 2 8]]


eval/accuracy,▁▆██
eval/loss,█▂▁▁
eval/macro_f1,▁▇██
eval/runtime,▂▁▄█
eval/samples_per_second,▇█▅▁
eval/steps_per_second,▇█▅▁
eval/weighted_f1,▁▇██
final/test_positive_f1,▁
final/test_positive_precision,▁
final/test_positive_recall,▁
+18,...


#### Run experiment 3: text + categorical + numerical

In [28]:
result_text_cat_num = run_experiment(
    experiment_name="roberta_text_categorical_numerical_os3",
    use_categorical=True,
    use_numerical=True
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.969067,1.014313,0.577320,0.265752,0.438118
2,0.690268,0.533691,0.783505,0.594191,0.786826
3,0.518592,0.452175,0.773196,0.619231,0.802328



Validation metrics: {'eval_loss': 0.4521751403808594, 'eval_accuracy': 0.7731958762886598, 'eval_macro_f1': 0.6192311825375595, 'eval_weighted_f1': 0.8023278557138036, 'eval_runtime': 0.3439, 'eval_samples_per_second': 564.106, 'eval_steps_per_second': 37.801, 'epoch': 3.0}

TRAIN CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.87      0.91      0.89       326
     neutral       0.92      0.73      0.81       433
    positive       0.36      1.00      0.53        42

    accuracy                           0.82       801
   macro avg       0.72      0.88      0.75       801
weighted avg       0.87      0.82      0.83       801


TRAIN CONFUSION MATRIX
[[298  28   0]
 [ 44 315  74]
 [  0   0  42]]



VAL CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.85      0.88      0.86        82
     neutral       0.88      0.69      0.77       109
    positive       0.12      1.00      0.22         3

    accuracy                           0.77       194
   macro avg       0.62      0.86      0.62       194
weighted avg       0.86      0.77      0.80       194


VAL CONFUSION MATRIX
[[72 10  0]
 [13 75 21]
 [ 0  0  3]]



TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.80      0.80      0.80        10
     neutral       0.67      0.60      0.63        10
    positive       0.82      0.90      0.86        10

    accuracy                           0.77        30
   macro avg       0.76      0.77      0.76        30
weighted avg       0.76      0.77      0.76        30


TEST CONFUSION MATRIX
[[8 2 0]
 [2 6 2]
 [0 1 9]]


eval/accuracy,▁███
eval/loss,█▂▁▁
eval/macro_f1,▁███
eval/runtime,▁▂▂█
eval/samples_per_second,█▇▇▁
eval/steps_per_second,█▇▇▁
eval/weighted_f1,▁███
final/test_positive_f1,▁
final/test_positive_precision,▁
final/test_positive_recall,▁
+18,...


#### Final summary table

In [29]:
summary_df = pd.DataFrame([
    {
        "experiment": result_text_cat["experiment_name"],
        "val_macro_f1": result_text_cat["val_metrics"]["eval_macro_f1"],
        "val_weighted_f1": result_text_cat["val_metrics"]["eval_weighted_f1"],
        "val_accuracy": result_text_cat["val_metrics"]["eval_accuracy"],
        "val_positive_precision": result_text_cat["val_positive"]["precision"],
        "val_positive_recall": result_text_cat["val_positive"]["recall"],
        "val_positive_f1": result_text_cat["val_positive"]["f1"]
    },
    {
        "experiment": result_text_num["experiment_name"],
        "val_macro_f1": result_text_num["val_metrics"]["eval_macro_f1"],
        "val_weighted_f1": result_text_num["val_metrics"]["eval_weighted_f1"],
        "val_accuracy": result_text_num["val_metrics"]["eval_accuracy"],
        "val_positive_precision": result_text_num["val_positive"]["precision"],
        "val_positive_recall": result_text_num["val_positive"]["recall"],
        "val_positive_f1": result_text_num["val_positive"]["f1"]
    },
    {
        "experiment": result_text_cat_num["experiment_name"],
        "val_macro_f1": result_text_cat_num["val_metrics"]["eval_macro_f1"],
        "val_weighted_f1": result_text_cat_num["val_metrics"]["eval_weighted_f1"],
        "val_accuracy": result_text_cat_num["val_metrics"]["eval_accuracy"],
        "val_positive_precision": result_text_cat_num["val_positive"]["precision"],
        "val_positive_recall": result_text_cat_num["val_positive"]["recall"],
        "val_positive_f1": result_text_cat_num["val_positive"]["f1"]
    }
])

summary_df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)

,experiment,val_macro_f1,val_weighted_f1,val_accuracy,val_positive_precision,val_positive_recall,val_positive_f1
0,roberta_text_categorical_os3,0.661093,0.829085,0.819588,0.200000,0.666667,0.307692
1,roberta_text_categorical_numerical_os3,0.619231,0.802328,0.773196,0.125000,1.000000,0.222222
2,roberta_text_numerical_os3,0.599640,0.806876,0.773196,0.086957,0.666667,0.153846
